# 1. NaiveRAG 시스템 구현

- "고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf" 를 바탕으로 NaiveRAG를 구현하고 Langfuse를 사용하여 추적해봅니다.

## 환경설정

In [2]:
import os

# 1. macOS OpenMP 충돌 방지 (반드시 최상단 실행)
os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

# 2. 토크나이저 병렬 처리 충돌 방지
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("환경 설정 완료. 이제 다음 셀들을 실행하세요.")

환경 설정 완료. 이제 다음 셀들을 실행하세요.


In [20]:
!pip install --upgrade langfuse langchain

In [3]:
from dotenv import load_dotenv

load_dotenv()

True

## RAG 기본 파이프라인(skeleton code)

In [4]:
# 라이브러리 임포트

from langchain_text_splitters import RecursiveCharacterTextSplitter     # 분할기
from langchain_community.document_loaders import PyMuPDFLoader          # 로더
from langchain_community.vectorstores import FAISS                      # 벡터스토어
from langchain_core.output_parsers import StrOutputParser              
from langchain_core.runnables import RunnablePassthrough                
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langfuse.callback import CallbackHandler

### 사전 준비단계

In [5]:
# 단계 1: 문서 로드(Load Documents)
loader = PyMuPDFLoader('/Users/apple/Team2-RAG-Project/고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf')
docs = loader.load()

len(docs)

297

In [6]:
# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)

In [7]:
len(split_documents)

475

In [8]:
print(split_documents[1].page_content)

-2- 
 
 
 
 
 
목   차
   
. 
Ⅰ사업개요
 ·················································································································· 
 4 
1. 
 
사업개요
 ·············································································································· 
 4 
2. 
 
사업배경
 ·············································································································· 
 4 
3. 
 
사업범위
 ·············································································································· 
 5 
4. 기대효과 ··············································································································· 
 7 
 
. 
Ⅱ현황 및 문제점
 ········································································································ 
 8 
1. 
 
일반현황


In [9]:
# 단계 3: 임베딩(Embedding) 생성
embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x1282803a0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x1282cb520>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version='', openai_api_base=None, openai_api_type='', openai_proxy='', embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [10]:
# 단계 4: DB 생성(Create DB) 및 저장
# 벡터스토어를 생성합니다.
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

In [11]:
# 단계 5: 검색기(Retreiever) 생성
# 문서에 포함되어 있는 정보를 검색하고 생성합니다.
retriever = vectorstore.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x13a4b1cf0>)

In [12]:
retriever.invoke("'지능형 검색' 기능을 고도화하기 위해 도입하려는 주요 검색 방식은 무엇인가요?",)

[Document(metadata={'source': '/Users/apple/Team2-RAG-Project/고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf', 'file_path': '/Users/apple/Team2-RAG-Project/고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf', 'page': 77, 'total_pages': 297, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': 'Draw', 'producer': 'LibreOffice 26.2.0.3 (AARCH64)', 'creationDate': "D:20260209132951+09'00'", 'modDate': '', 'trapped': ''}, page_content='검색결과를제공하는서비스\n- \n  \n  \n  \n  \n  \n  \n  \n  \n  \n  \n \n최종지능형검색도입방식은적용가능성및활용도가높은서비스를\n \n \n결정하여도입함\n- AI선배,  \n  \n  \n  \n  \n  \n  \n  \n  \n  \n \n챗봇등검색과유사한작동을하는교내서비스와연동되어야함\n산출정보 \n• \n  \n프로그램목록,  프로그램명세서,  화면정의서,  \n  \n \n사용자매뉴얼등'),
 Document(metadata={'source': '/Users/apple/Team2-RAG-Project/고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf', 'file_path': '/Users/apple/Team2-RAG-Project/고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf', 'page': 77, 'total_pages': 297, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': 'Draw', 'producer': 'L

In [13]:
# 단계 6: 프롬프트 생성(Create Prompt)
# 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(
    """You are an assistant for question-answering tasks.
Use the following poeces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Answer in Korean.

#Question:
{question}
#Context:
{context}

#Answer:"""
)

# 단계 7: 언어모델(LLM) 생성
# 모델(LLM) 를 생성합니다.
llm = ChatOpenAI(model_name="gpt-5-mini", temperature=1)

# 단계 8: 체인(Chain) 생성
langfuse_handler = CallbackHandler() 
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [25]:
# 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "'지능형 검색' 기능을 고도화하기 위해 도입하려는 주요 검색 방식은 무엇인가요?"
responce = chain.invoke(question,
                        config={"callbacks": [langfuse_handler]})
print(responce)

문서에 따르면 지능형 검색 고도화를 위해 도입하려는 주요 검색 방식은 다음 네 가지입니다.

- 의미기반 검색: 사용자가 입력한 키워드의 의미와 문맥을 분석해 의도에 맞는 결과 제공  
- 개인화 검색/추천검색: 사용자 특성·선호·상황을 반영한 맞춤형 검색 및 추천  
- 유사문장 검색: 입력 문장과 데이터베이스 말뭉치의 유사 문장을 찾아 유사도 순으로 제공  
- 다국어 검색: 영문 입력에 대해 한국어 문장과 동일한 검색결과를 제공하는 기능


In [15]:
retrieved_chunks = retriever.invoke("'지능형 검색' 기능을 고도화하기 위해 도입하려는 주요 검색 방식은 무엇인가요?")
for i, doc in enumerate(retrieved_chunks):
    page_num = doc.metadata.get("page", "정보 없음")
    print(page_num) 

77
77
179
171


In [24]:
import os
from langfuse import Langfuse
from dotenv import load_dotenv

load_dotenv()

# 환경 변수 직접 출력해서 확인 (디버깅용)
print(f"Host: {os.getenv('LANGFUSE_HOST')}") 

langfuse = Langfuse()
trace = langfuse.trace(name="Gildong_Connection_Test")
print("Trace 생성됨")

# 강제 전송
langfuse.flush()
print("전송 시도 완료. 이제 홈페이지를 새로고침 해보세요!")

Host: https://us.cloud.langfuse.com
Trace 생성됨
전송 시도 완료. 이제 홈페이지를 새로고침 해보세요!


In [16]:
question = "'성적확정관리' 요구사항 중, 학생이 성적을 조회하기 위해 반드시 선행해야 하는 조건은 무엇인가요?"
responce = chain.invoke(question,
                        config={"callbacks": [langfuse_handler]})
print(responce)

학생이 성적을 조회하려면 교수자가 성적입력을 완료하고 해당 과목을 성적잠금(성적확정) 처리해야 합니다.


In [17]:
retrieved_chunks = retriever.invoke("'성적확정관리' 요구사항 중, 학생이 성적을 조회하기 위해 반드시 선행해야 하는 조건은 무엇인가요?")
for i, doc in enumerate(retrieved_chunks):
    page_num = doc.metadata.get("page", "정보 없음")
    print(page_num) 

108
107
106
108


#### 평가 데이터셋
Q. 이번 차세대 시스템 구축 사업에서 '지능형 검색' 기능을 고도화하기 위해 도입하려는 주요 검색 방식 3가지는 무엇인가요?	

A. 의미기반 검색(문맥 분석), 개인화/추천 검색(사용자 맞춤형 정보 제공), 유사문장 검색(유사도가 높은 최적 문장 제공) 등입니다. 

-78페이지-

Q. 차세대 학사행정시스템의 '성적확정관리' 요구사항 중, 학생이 성적을 조회하기 위해 반드시 선행해야 하는 조건은 무엇인가요?	

A.학생은 수강소감 설문응답을 완료해야만 성적 조회가 가능합니다. (단, 소속 학과에 따라 안전교육 이수 등 추가 조건이 있을 수 있음)

-109페이지-